# 01 - Eksploracja danych

Cel: zrozumieć co mamy - ile wierszy, jakie kolumny, typy, zakresy dat, kompletność.

## 1. Import bibliotek

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## 2. Wczytanie danych

In [3]:
measurements = pd.read_csv('../data/raw/Measurements.csv') 
travel_time_a = pd.read_csv('../data/raw/Travel_Time_A.csv')
travel_time_b = pd.read_csv('../data/raw/Travel_Time_B.csv')
weather = pd.read_csv('../data/raw/Weather_Conditions.csv')
route_details = pd.read_csv('../data/raw/Route_Details.csv')

## 3. Przegląd każdej tabeli

In [4]:
print("Ilość wierszy i kolumn w tabeli Measurements:", measurements.shape)
print("\nInformacje o tabeli Measurements:")
print(measurements.info())

Ilość wierszy i kolumn w tabeli Measurements: (44588, 6)

Informacje o tabeli Measurements:
<class 'pandas.DataFrame'>
RangeIndex: 44588 entries, 0 to 44587
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Measurement_Id    44588 non-null  int64
 1   Route_Id          44588 non-null  int64
 2   Direction         44588 non-null  str  
 3   Date              44588 non-null  str  
 4   Day_Of_Week       44588 non-null  str  
 5   Measurement_Time  44588 non-null  str  
dtypes: int64(2), str(4)
memory usage: 2.0 MB
None


In [5]:
route_details

,Route_Id,Direction,Start_Of_Route,End_Of_Route,Description
0,1,A,Tarnowskie Góry Dworzec,Katowice Piotra Skargi,Przez Gliwice
1,2,A,Tarnowskie Góry Dworzec,Katowice Piotra Skargi,Przez Bytom
2,3,A,Tarnowskie Góry Dworzec,Katowice Piotra Skargi,Przez Piekary Śląskie
3,4,A,Tarnowskie Góry Dworzec,Katowice Piotra Skargi,Przez Siemianowice Śląskie
4,5,A,Tarnowskie Góry Dworzec,Katowice Piotra Skargi,Przez Pyrzowice
5,1,B,Katowice Piotra Skargi,Tarnowskie Góry Dworzec,Przez Gliwice
6,2,B,Katowice Piotra Skargi,Tarnowskie Góry Dworzec,Przez Bytom
7,3,B,Katowice Piotra Skargi,Tarnowskie Góry Dworzec,Przez Piekary Śląskie
8,4,B,Katowice Piotra Skargi,Tarnowskie Góry Dworzec,Przez Siemianowice Śląskie
9,5,B,Katowice Piotra Skargi,Tarnowskie Góry Dworzec,Przez Pyrzowice


In [6]:
print("Ilość wierszy i kolumn w tabeli Travel_Time_A:", travel_time_a.shape)
print("Ilość wierszy i kolumn w tabeli Travel_Time_B:", travel_time_b.shape)

print("\nOpis statystyczny tabeli Travel_Time_A:")
print(travel_time_a[['Duration_Min', 'Delay_Min']].describe().round(2)) 

print("\nOpis statystyczny tabeli Travel_Time_B:")
print(travel_time_b[['Duration_Min', 'Delay_Min']].describe().round(2))


Ilość wierszy i kolumn w tabeli Travel_Time_A: (22299, 3)
Ilość wierszy i kolumn w tabeli Travel_Time_B: (22303, 3)

Opis statystyczny tabeli Travel_Time_A:
       Duration_Min  Delay_Min
count      22295.00   22295.00
mean          44.97       6.57
std            5.31       5.03
min           35.00       0.00
25%           41.00       2.00
50%           44.00       5.00
75%           49.00      10.00
max           76.00      39.00

Opis statystyczny tabeli Travel_Time_B:
       Duration_Min  Delay_Min
count      22293.00   22293.00
mean          42.26       6.26
std            6.46       5.88
min           32.00       0.00
25%           38.00       2.00
50%           41.00       4.00
75%           45.00       9.00
max           86.00      47.00


In [7]:
print("Ilość wierszy i kolumn w tabeli Weather:", weather.shape)
print("\nInformacje o tabeli Weather:")
print(weather.info())

print("Średnia wartość Snow_Origin:", weather['Snow_Origin'].mean().round(2))
print("Średnia wartość Snow_Destination:", weather['Snow_Destination'].mean().round(2))

weather['Visibility_Origin'].describe().round(2)
low_visibility = weather[weather['Visibility_Origin'] < 10000]['Visibility_Origin'].count()
print("Ilość wierszy z Visibility_Origin < 10000:", low_visibility)
print((low_visibility / len(weather) * 100).round(2), "%")

Ilość wierszy i kolumn w tabeli Weather: (44588, 13)

Informacje o tabeli Weather:
<class 'pandas.DataFrame'>
RangeIndex: 44588 entries, 0 to 44587
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Measurement_Id           44588 non-null  int64  
 1   Weather_Origin           44588 non-null  str    
 2   Temperature_Origin       44588 non-null  float64
 3   Humidity_Origin          44588 non-null  int64  
 4   Rain_Origin              44588 non-null  float64
 5   Snow_Origin              44588 non-null  int64  
 6   Visibility_Origin        44588 non-null  int64  
 7   Weather_Destination      44588 non-null  str    
 8   Temperature_Destination  44588 non-null  float64
 9   Humidity_Destination     44588 non-null  int64  
 10  Rain_Destination         44588 non-null  float64
 11  Snow_Destination         44588 non-null  int64  
 12  Visibility_Destination   44588 non-null  int64  
dtypes: f

Kolumny Snow_Origin oraz Snow_Destination okazują się zbędne, jako że pomiary zostały zebrane w lipcu. 

Kolumny Visibility_Origin oraz Visibility_Destination są warte uwagi podczas sprawdzania korealacji z Duration_Min, ponieważ wartości z Visibility < 10000 jest jedynie 6.12%. Przy tak małej wariancji korelacja z Duration_Min będzie prawdopodobnie bliska zeru. 

## 4. Zakres dat

In [8]:
measurements['Date'] = pd.to_datetime(measurements['Date'], format='%Y-%m-%d')
print("Początek pomiarów:", measurements['Date'].min())
print("Koniec pomiarów:", measurements['Date'].max())

Początek pomiarów: 2025-07-01 00:00:00
Koniec pomiarów: 2025-07-31 00:00:00


## 5. Kompletność danych

In [9]:
measurements_nulls = measurements.isnull().sum()
travel_time_a_nulls = travel_time_a.isnull().sum()
travel_time_b_nulls = travel_time_b.isnull().sum()
weather_nulls = weather.isnull().sum()

print("Braki danych w tabeli Measurements:")
print(measurements_nulls)
print("\nBraki danych w tabeli Travel_Time_A:")
print(travel_time_a_nulls)
print("\nBraki danych w tabeli Travel_Time_B:")
print(travel_time_b_nulls)
print("\nBraki danych w tabeli Weather:")
print(weather_nulls)

Braki danych w tabeli Measurements:
Measurement_Id      0
Route_Id            0
Direction           0
Date                0
Day_Of_Week         0
Measurement_Time    0
dtype: int64

Braki danych w tabeli Travel_Time_A:
Measurement_Id    0
Duration_Min      4
Delay_Min         4
dtype: int64

Braki danych w tabeli Travel_Time_B:
Measurement_Id     0
Duration_Min      10
Delay_Min         10
dtype: int64

Braki danych w tabeli Weather:
Measurement_Id             0
Weather_Origin             0
Temperature_Origin         0
Humidity_Origin            0
Rain_Origin                0
Snow_Origin                0
Visibility_Origin          0
Weather_Destination        0
Temperature_Destination    0
Humidity_Destination       0
Rain_Destination           0
Snow_Destination           0
Visibility_Destination     0
dtype: int64


In [10]:

print(travel_time_a['Measurement_Id'].min())
print(travel_time_a['Measurement_Id'].max())
print(travel_time_a.head())
print(travel_time_a.tail())

1
44588
   Measurement_Id  Duration_Min  Delay_Min
0               1          42.0        2.0
1               2          39.0        2.0
2               3          39.0        4.0
3               4          43.0        3.0
4               5          44.0        4.0
       Measurement_Id  Duration_Min  Delay_Min
22294           44584          42.0        2.0
22295           44585          39.0        2.0
22296           44586          39.0        4.0
22297           44587          44.0        4.0
22298           44588          44.0        4.0


In [11]:
print(travel_time_a['Measurement_Id'] % 10)

0        1
1        2
2        3
3        4
4        5
        ..
22294    4
22295    5
22296    6
22297    7
22298    8
Name: Measurement_Id, Length: 22299, dtype: int64


Podczas analizy wstępnej zauważyłem ciekawy przypadek w tabeli Travel_Time_A - w pewnym momencie wartości w kolumnie Measurement_Id zamiast kończyć się 1, 2, 3, 4, 5, przesunęły się najpierw o jedną wartość - 0, 1, 2, 3, 4 aby na samym końcu zbioru kończyć się wartościami 4, 5, 6, 7, 8.

Anomalia ta prawdopodobnie wynika z błędu w kodzie który zbierał dane i zapisywał Measurement_Id. Sprawdzam:

In [12]:
print(travel_time_b['Measurement_Id'].min())
print(travel_time_b['Measurement_Id'].max())
print(travel_time_b.head())
print(travel_time_b.tail())

6
44593
   Measurement_Id  Duration_Min  Delay_Min
0               6          39.0        2.0
1               7          33.0        1.0
2               8          37.0        3.0
3               9          41.0        3.0
4              10          43.0        4.0
       Measurement_Id  Duration_Min  Delay_Min
22298           44589          39.0        2.0
22299           44590          33.0        1.0
22300           44591          37.0        3.0
22301           44592          41.0        3.0
22302           44593          43.0        4.0


In [13]:
print(travel_time_b['Measurement_Id'] % 10)

0        6
1        7
2        8
3        9
4        0
        ..
22298    9
22299    0
22300    1
22301    2
22302    3
Name: Measurement_Id, Length: 22303, dtype: int64


In [14]:
print(measurements[measurements['Direction'] == 'A']['Direction'].count())
print(measurements[measurements['Direction'] == 'B']['Direction'].count())

22295
22293


Sprawdziłem czy anomalia ta występuje także w tabeli Travel_Time_B, z czego wyszło że on także przesunął się o taką samą ilość wierszy. Powoduje to potrzebę skontrolowania czy tabela Measurements dobrze dopasowuje trasę do czasu przejazdu. 

Tabela Measurements poza Measurement_Id posiada także kolumnę Direction dzięki której będzie można sprawdzić czy wszystkie wiersze się pokryją, czy wystąpił jakiś większy błąd w zbiorze. 

Pierwszym krokiem który podjąłem było sprawdzenie przy którym Measurement_Id pierwszy raz wystąpiło przesunięcie - 6145 było ostatnim z poprawną końcówką "5", kolejna seria zaczęła się już od 6150 i skończyła na 6154 i faktycznie możemy takie przesunięcie zauważyć także w tabeli Measurements. 

Widać tam jasno że kolejna seria z kierunku "B" posiada tylko cztery rekordy - 6146, 6147, 6148 oraz 6149, które spowodowały że kolejna seria kierunku "A" zaczęła się od wcześniejszego ID. Szukam wyjaśnienia w tabeli Logs. 

Problem znaleziony. Tego dnia o tej porze, przy jednym z rekordów widnieje log: 
2025-07-05 06:20,ERROR,Error 5_B: HTTPSConnectionPool(host='api.tomtom.com', port=443): Max retries exceeded with url: (...)

Jest to realny ubytek danych, który wyjaśnia dlaczego kierunku "A" w Measurements jest 22295 a kierunku "B" jedynie 22293

## 6. Pytania analityczne

1. Czy występuje różnica czasu przejazdu w kierunku A oraz B? Jeśli tak, w jakich godzinach jest realnie widoczna?
2. Czy pogoda wpływa realnie na czas przejazdu? Jaka jest korelacja pomiędzy kolumnami Weather_Conditions a Duration_Min?
3. Która z tras ma najniższy średni czas, a na której mamy większą pewność że długość przejazdu znacznie się nie odchyli?
4. O których godzinach czas przejazdu jest minimalny, a kiedy maksymalny? Czy są to codziennie te same pory?
5. Czy weekend znacznie wpływa na czas przejazdu? 